####  Installing packages

In [ ]:
!pip install transformers==4.41.2

!pip install torch
!pip install sentencepiece
!pip install accelerate

!pip install langchain
!pip install langchain-community
!pip install langchain-text-splitters
!pip install faiss-cpu
!pip install pypdf
!pip install sentence-transformers

#### Importing necessary libraries

In [2]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from transformers import pipeline

#### Loading PDF and Chunking

In [ ]:
# Path to the PDF file to be loaded
pdf_path = "C:\\Users\\issac\\Documents\\Next GenAI assignment\\Assignment 6\\introtoml.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("PDF Loaded Successfully")
print("Total Pages:", len(documents))

PDF Loaded Successfully
Total Pages: 392


In [4]:
print(documents[0].page_content[:3000])

Andreas C. Müller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y t h o n   
A GUIDE FOR DATA SCIENTISTS


In [ ]:
# Chunking the text into smaller pieces for better retrieval performance
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

texts = text_splitter.split_documents(documents)

print("Total Chunks Created:", len(texts))

Total Chunks Created: 1680


#### Loading Embedded model

In [ ]:
# Creating the embedding model using HuggingFace's sentence-transformers
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded Successfully")

C:\Users\issac\AppData\Local\Temp\ipykernel_2700\893028751.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
C:\Users\issac\ragnv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embedding Model Loaded Successfully


#### Create FAISS DB and storing

In [ ]:
# Creating the FAISS vector database from the chunked texts and their embeddings
vector_store = FAISS.from_documents(
    texts,
    embedding_model
)

print("FAISS Vector Database Created Successfully")

FAISS Vector Database Created Successfully


In [ ]:
# Saving the vector database locally for future use
vector_store.save_local("faiss_index")

print("Vector DB Saved Successfully")

Vector DB Saved Successfully


#### Retrieval

In [ ]:
# Loading the vector database from local storage
def retrieve_docs(query, k=2):
    docs = vector_store.similarity_search(query, k=k)
    return docs

#### Loading Q&A Model

In [ ]:
# Creating the question-answering pipeline using a pre-trained model from HuggingFace
qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)

print("Question Answering Model Loaded Successfully")

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

C:\Users\issac\ragnv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\issac\.cache\huggingface\hub\models--distilbert-base-cased-distilled-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Question Answering Model Loaded Successfully


In [ ]:
# Example query to retrieve relevant chunks and answer the question
query = "What is supervised learning?"

retrieved_docs = retrieve_docs(query, k=2)

for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Retrieved Chunk {i+1} ---\n")
    print(doc.page_content[:1500])


--- Retrieved Chunk 1 ---

spam.
Machine learning algorithms that learn from input/output pairs are called supervised
learning algorithms because a “teacher” provides supervision to the algorithms in the
form of the desired outputs for each example that they learn from. While creating a
dataset of inputs and outputs is often a laborious manual process, supervised learning
algorithms are well understood and their performance is easy to measure. If your

--- Retrieved Chunk 2 ---

46 | Chapter 2: Supervised Learning


#### RAG Function

In [ ]:
# Function to perform RAG: retrieve relevant chunks, extract short answer, and provide context-based explanation
def ask_rag(question, k=3):
    docs = retrieve_docs(question, k=k)

    # Combine retrieved chunks
    context = "\n\n".join([doc.page_content for doc in docs])

    # Extract short answer using QA model
    result = qa_pipeline(
        question=question,
        context=context
    )

    short_answer = result["answer"]

    # Use first part of retrieved context as explanation support
    explanation = context[:500]

    final_answer = f"""
Answer:
{short_answer}

Context-Based Explanation:
{explanation}
"""

    return final_answer

In [ ]:
# Example question to test the RAG system
question = "Explain overfitting in machine learning"

answer = ask_rag(question, k=2)

print("Question:", question)
print("\nAnswer:\n")
print(answer)

Question: Explain overfitting in machine learning

Answer:


Answer:
models that are very complex and highly overfit to the training data

Context-Based Explanation:
occurs when you fit a model too closely to the particularities of the training set and
obtain a model that works well on the training set but is not able to generalize to new
data. On the other hand, if your model is too simple—say, “Everybody who owns a
house buys a boat”—then you might not be able to capture all the aspects of and vari‐
ability in the data, and your model will do badly even on the training set. Choosing
too simple a model is called underfitting.

leads to models that are very 



In [ ]:
# Testing with multiple questions
questions = [
    "What is supervised learning?",
    "What is overfitting?",
    "Explain neural networks",
    "What is gradient descent?"
]

for q in questions:
    print("\n============================")
    print("QUESTION:", q)
    print("============================\n")

    print(ask_rag(q, k=5))
    print("\n")


QUESTION: What is supervised learning?


Answer:
Machine learning algorithms that learn from input/output pairs

Context-Based Explanation:
spam.
Machine learning algorithms that learn from input/output pairs are called supervised
learning algorithms because a “teacher” provides supervision to the algorithms in the
form of the desired outputs for each example that they learn from. While creating a
dataset of inputs and outputs is often a laborious manual process, supervised learning
algorithms are well understood and their performance is easy to measure. If your

46 | Chapter 2: Supervised Learning

26 | Chapter 2: Supervised Learni




QUESTION: What is overfitting?


Answer:
Choosing
too simple a model

Context-Based Explanation:
occurs when you fit a model too closely to the particularities of the training set and
obtain a model that works well on the training set but is not able to generalize to new
data. On the other hand, if your model is too simple—say, “Everybody who owns a
ho

In [33]:
# Interactive loop for user to ask questions
while True:
    user_question = input("\nAsk your ML question (type 'exit' to stop): ")

    if user_question.lower() == "exit":
        print("Exiting RAG System...")
        break

    answer = ask_rag(user_question, k=2)

    print("\n============================")
    print("QUESTION:", user_question)
    print("============================\n")

    print("ANSWER:\n")
    print(answer)
    print("\n")


QUESTION: What is Machine Learning?

ANSWER:


Answer:
extracting knowledge from data

Context-Based Explanation:
CHAPTER 1
Introduction
Machine learning is about extracting knowledge from data. It is a research field at the
intersection of statistics, artificial intelligence, and computer science and is also
known as predictive analytics or statistical learning. The application of machine
learning methods has in recent years become ubiquitous in everyday life. From auto‐
matic recommendations of which movies to watch, to what food to order or which

spam.
Machine learning algorithms that learn from input/o




QUESTION: What is Deep learning?

ANSWER:


Answer:
Neural Networks

Context-Based Explanation:
flexible interface to build neural networks and track the rapid progress in deep learn‐
ing research. All of the popular deep learning libraries also allow the use of high-
performance graphics processing units (GPUs), which scikit-learn does not
support. Using GPUs allows us to acce